# Пример 2. Декодирование и воспроизводимость

**Модуль I · Тема 2** — БЯМ как вычислительное ядро агента

Модель не «выбирает ответ». На каждом шаге она получает **распределение вероятностей** по всему словарю и берёт из него один токен. *Как именно* берёт — задают параметры декодирования: `temperature`, `top_p`, `seed`.

Для агента это не косметика: от этих параметров зависит, **воспроизводим** ли результат. Роль-классификатор и роль-редактор требуют противоположных настроек.

### Что показано
1. Как выглядит само распределение вероятностей (logprobs).
2. Температура: от детерминированности к разбросу.
3. Почему `temperature=0` не гарантирует строгой воспроизводимости.
4. `top_p` и отсечение «хвоста» распределения.
5. Разные типы задач требуют разных параметров.

### Доступ к модели
Параметры — из переменных окружения `LLM_API_KEY`, `LLM_BASE_URL`, `LLM_MODEL`; ключ в код не вставляется.

In [1]:
import os, time, statistics
from collections import Counter
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("LLM_API_KEY"),
                base_url=os.environ.get("LLM_BASE_URL"))
MODEL = os.environ.get("LLM_MODEL", "openai/gpt-4.1-mini")

def ask(prompt, n=1, **kw):
    """Один вызов модели. Возвращает список из n текстов ответа."""
    r = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}], n=n, **kw)
    return [c.message.content.strip() for c in r.choices]

print("Модель:", MODEL)
print("Проверка:", ask("Ответь одним словом: столица Франции?", temperature=0, max_tokens=10)[0])

Модель: openai/gpt-4.1-mini


Проверка: Париж


## 1. Под капотом: распределение вероятностей

Запросим `logprobs` — логарифмы вероятностей токенов-кандидатов на первом шаге генерации. Это позволяет увидеть то, из чего модель выбирает.

- `logprobs=True` — вернуть вероятности выбранных токенов;
- `top_logprobs=k` — вернуть ещё и k лучших альтернатив на каждом шаге.

In [2]:
import math

r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Закончи фразу одним словом: Столица Франции —"}],
    temperature=0, max_tokens=5, logprobs=True, top_logprobs=5)

first = r.choices[0].logprobs.content[0]
print(f"Выбран токен: {first.token!r}\n")
print(f"{'кандидат':>14} | {'logprob':>9} | вероятность")
print("-" * 44)
for alt in first.top_logprobs:
    print(f"{alt.token!r:>14} | {alt.logprob:9.4f} | {math.exp(alt.logprob):.4%}")

Выбран токен: 'П'

      кандидат |   logprob | вероятность
--------------------------------------------
           'П' |   -0.0002 | 99.9771%
          'Ст' |   -8.6252 | 0.0180%
         'Пар' |  -10.6252 | 0.0024%
           'п' |  -10.7502 | 0.0021%
       'Paris' |  -13.8752 | 0.0001%


Так выглядит «мышление» модели на одном шаге: не выбор смысла, а ранжирование токенов по вероятности. Все параметры декодирования — это правила отбора **из этого списка**.

## 2. Температура

Температура масштабирует распределение перед выбором:
- **низкая** (→0) — распределение заостряется, почти всегда берётся самый вероятный токен;
- **высокая** (>1) — распределение сглаживается, у редких токенов появляется шанс.

Замерим воспроизводимость: один и тот же промпт по 5 раз при разных температурах.

In [3]:
PROMPT = "Придумай название для сервиса доставки еды. Ответь только названием."
REPEATS = 5

rows = []
for t in [0.0, 0.3, 0.7, 1.2]:
    answers = []
    for _ in range(REPEATS):
        answers += ask(PROMPT, temperature=t, max_tokens=20)
    uniq = len(set(answers))
    top = Counter(answers).most_common(1)[0]
    rows.append({"temperature": t, "уникальных из 5": uniq,
                 "самый частый": top[0][:28], "повторов": top[1]})

import pandas as pd
print(pd.DataFrame(rows).to_string(index=False))

 temperature  уникальных из 5 самый частый  повторов
         0.0                1   ВкусВперёд         5
         0.3                2   ВкусВперёд         4
         0.7                4   ВкусВперед         2
         1.2                4 ВкусЭкспресс         2


Чем выше температура, тем больше уникальных ответов на один и тот же запрос. Это и есть практическое определение воспроизводимости.

## 3. Почему `temperature=0` — не гарантия детерминизма

Частое заблуждение: «поставил ноль — получишь всегда одно и то же». На практике совпадение очень вероятно, но **не гарантировано**.

Причины лежат вне вашего кода:
- параллельные вычисления на GPU дают неассоциативность сложения чисел с плавающей точкой — порядок суммирования влияет на младшие разряды;
- батчинг запросов на стороне провайдера меняет этот порядок;
- модель за одним и тем же именем может быть обновлена;
- маршрутизация запроса может попасть на другое оборудование.

Проверим на **неоднозначном** промпте, где вероятности верхних кандидатов близки — именно там нестабильность проявляется чаще всего.

In [4]:
AMBIG = "Назови одно прилагательное к слову «море». Ответь одним словом."
res = [ask(AMBIG, temperature=0, max_tokens=10)[0] for _ in range(8)]

print("8 запусков при temperature=0:")
for i, a in enumerate(res, 1):
    print(f"  {i}. {a}")
print("\nуникальных ответов:", len(set(res)))
print("вывод:", "детерминизм соблюдён" if len(set(res)) == 1
      else "детерминизм НЕ соблюдён — совпадение не гарантируется")

8 запусков при temperature=0:
  1. бескрайнее
  2. бескрайнее
  3. бескрайнее
  4. бескрайнее
  5. бескрайнее
  6. бескрайнее
  7. бескрайнее
  8. бескрайнее

уникальных ответов: 1
вывод: детерминизм соблюдён


**Как читать этот результат.** В данном прогоне все 8 ответов совпали — это типичная ситуация: при `temperature=0` расхождения возникают **редко**. Именно поэтому они опасны: разработчик видит стабильность на десятке ручных проверок и закладывается на неё, а расхождение всплывает позже — на другом оборудовании, при нагрузке или после обновления модели.

Правильная формулировка: `temperature=0` даёт **высокую вероятность** совпадения, но не контрактную гарантию. Для регрессионных тестов (модуль IV) этого недостаточно.

**Практический вывод для агента.** Если требуется строгая воспроизводимость (тесты, регрессия — модуль IV), нельзя полагаться только на `temperature=0`. Дополнительно применяют:
- `seed` (если провайдер поддерживает) — фиксация псевдослучайности;
- ограничение выбора форматом: строгий перечень допустимых ответов, структурированный вывод (модуль II);
- кэширование ответов на одинаковые запросы.

In [5]:
# Параметр seed: поддерживается не всеми провайдерами. Проверим доступность.
try:
    a = ask("Назови случайное число от 1 до 100. Только число.", temperature=1.0, seed=42, max_tokens=10)[0]
    b = ask("Назови случайное число от 1 до 100. Только число.", temperature=1.0, seed=42, max_tokens=10)[0]
    print(f"seed=42 -> {a!r} и {b!r} | совпали: {a == b}")
except Exception as e:
    print("Провайдер не поддерживает seed:", type(e).__name__, str(e)[:100])

seed=42 -> '47' и '47' | совпали: True


## 4. `top_p` — отсечение хвоста распределения

`top_p` (nucleus sampling) оставляет минимальный набор самых вероятных токенов, суммарная вероятность которых достигает `p`, и выбирает только из него.

- `top_p=1.0` — доступен весь словарь;
- `top_p=0.1` — только «ядро» самых вероятных вариантов.

Это независимый от температуры способ ограничить разброс.

In [6]:
rows = []
for p in [0.1, 0.5, 1.0]:
    answers = []
    for _ in range(REPEATS):
        answers += ask(PROMPT, temperature=1.0, top_p=p, max_tokens=20)
    rows.append({"top_p": p, "temperature": 1.0, "уникальных из 5": len(set(answers)),
                 "пример": answers[0][:30]})
print(pd.DataFrame(rows).to_string(index=False))

 top_p  temperature  уникальных из 5     пример
   0.1          1.0                1 ВкусВперёд
   0.5          1.0                2 ВкусВперед
   1.0          1.0                5 Вкусомания


## 5. Разным ролям — разные параметры

Ключевой вывод темы. Возьмём две задачи противоположного типа и посмотрим, какие настройки для них уместны.

In [7]:
CLASSIFY = ("Классифицируй обращение в поддержку строго одним словом из списка: "
            "[доставка, оплата, возврат, качество].\n"
            "Обращение: «Товар пришёл разбитым, хочу вернуть деньги»")
CREATIVE = "Придумай слоган для службы поддержки интернет-магазина. Только слоган."

for name, prompt, temp in [("классификация", CLASSIFY, 0.0), ("классификация", CLASSIFY, 1.2),
                           ("генерация",     CREATIVE, 0.0), ("генерация",     CREATIVE, 1.2)]:
    ans = [ask(prompt, temperature=temp, max_tokens=30)[0] for _ in range(3)]
    print(f"[{name:13} t={temp}] уникальных: {len(set(ans))} | {ans[0][:60]}")

[классификация t=0.0] уникальных: 1 | возврат


[классификация t=1.2] уникальных: 1 | возврат


[генерация     t=0.0] уникальных: 1 | Всегда рядом, чтобы помочь!


[генерация     t=1.2] уникальных: 3 | Всегда рядом, чтобы помочь!


**Обратите внимание на строку классификации при `t=1.2`:** ответ остался единственным («возврат»), хотя температура высокая.

Это не противоречие, а важный нюанс. Температура сглаживает распределение, но если оно **изначально очень острое** (правильный ответ очевиден, его вероятность близка к 100 % — как в примере с logprobs выше), то даже сильное сглаживание не выводит вперёд другого кандидата.

Отсюда практический вывод: высокая температура опасна не везде одинаково — она проявляется на **неоднозначных** входах. Поэтому проверять устойчивость роли нужно на пограничных и неполных данных, а не на очевидных примерах.

**Как это переносится в проект.**

| Тип роли | Требование | Температура | Почему |
|----------|-----------|:-----------:|--------|
| Классификатор, маршрутизатор, извлечение данных | одинаковый вход → одинаковый выход | **0** | разброс здесь = ошибка; ответ сверяется с эталоном |
| Агент-исполнитель с инструментами | предсказуемый выбор инструмента | **0–0,3** | недетерминированность ломает воспроизводимость трасс |
| Генерация текста, черновики, варианты | нужно разнообразие | **0,7–1,0** | одинаковые ответы обесценивают роль |
| Агент-критик | стабильность оценки | **0** | оценка должна быть повторяемой (модуль IV) |

Эту таблицу студент заполняет для **своих** ролей в лабораторной работе ([КИМ-1.2](../../kim-lab-02.md), часть 2) и переносит в требования к агентам.

## Итоги

- Модель на каждом шаге ранжирует токены по вероятности; параметры декодирования — правила отбора из этого списка.
- **Температура** управляет разбросом: чем выше, тем меньше воспроизводимость.
- **`temperature=0` не даёт строгих гарантий** — источники недетерминизма лежат на стороне вычислений и провайдера; для тестов нужны дополнительные меры.
- **`top_p`** ограничивает разброс независимо от температуры.
- Параметры декодирования выбираются **под роль**, а не «для проекта в целом».